Notebook for the indexing pipeline.

In [1]:
# imports
import json
import re

In [2]:
# read in the raw recipes
path_raw = "../recipe_crawler/recipes.json"
with open(path_raw, 'r') as f:
    recipes = json.load(f)

In [3]:
# clean instruction group name
def clean_instruction_group_name(name):
    """Normalise instruction group name."""
    if name is None or not isinstance(name, str):
        return name
    
    original = name
    
    # 1. Strip wrapping parentheses: (FULL RECIPE) -> FULL RECIPE
    if name.startswith("(") and name.endswith(")"):
        name = name[1:-1]
    
    # 2. Strip trailing characters: colons, quotes, exclamation, dots, whitespace
    name = name.rstrip(':\'"!., ')
    
    # 3. Strip leading characters: quotes, exclamation, whitespace
    name = name.lstrip('\'"!., ')
    
    # 4. Remove stray quotes inside the name
    name = name.replace('"', '').replace('“', '').replace('”', '').replace("'", "")
    
    # 5. Collapse multiple spaces
    name = re.sub(r'\s+', ' ', name).strip()
    
    # 6. Title case
    name = name.title()
    
    return name if name else original  # If we stripped everything, keep original

# Process and track changes
changes = []

for recipe in recipes:
    for group in recipe["instruction_groups"]:
        old_name = group["group_name"]
        new_name = clean_instruction_group_name(old_name)
        
        if old_name != new_name:
            changes.append((old_name, new_name, recipe["recipe_name"]))
        
        group["group_name"] = new_name

# Save
with open("recipes_cleaned.json", "w") as f:
    json.dump(recipes, f, indent=2, ensure_ascii=False)

# load the cleaned version for futhre processing
with open("recipes_cleaned.json", 'r') as f:
    recipes = json.load(f)

In [4]:
# replace instruction group with junk group name to "Full Recipe" (where it is the only instruction group in the recipe)
junk_values = {None, '', '.', '<br>', ' '}

for r in recipes:
    groups = r["instruction_groups"]
    
    if len(groups) == 1 and groups[0]["group_name"] in junk_values:
        old = groups[0]["group_name"]
        groups[0]["group_name"] = "Full Recipe"

# Save
with open("recipes_cleaned.json", "w") as f:
    json.dump(recipes, f, indent=2, ensure_ascii=False)

In [5]:
# for recipes with multpile instruction groups, with single or multipe junk group names
# this can be replaced to "Other Component" (Same as saying all junk group names can be 
# replaced to "Other Component"). 
# This does mean that instruction groups will not have unique names for each respective group.

junk_values = {None, '', '.', '<br>', ' ', '<Br>'}

for r in recipes:
    groups = r["instruction_groups"]
    
    # Only act on recipes with multiple instruction groups
    if len(groups) > 1:
        for g in groups:
            if g["group_name"] in junk_values:
                old = repr(g["group_name"])
                g["group_name"] = "Unknown"
# Save
with open("recipes_cleaned.json", "w") as f:
    json.dump(recipes, f, indent=2, ensure_ascii=False)

In [6]:
# remove instruction group with no instructions
for r in recipes:
    before = len(r["instruction_groups"])
    r["instruction_groups"] = [g for g in r["instruction_groups"] if len(g["instructions"]) > 0]
    after = len(r["instruction_groups"])

# Save
with open("recipes_cleaned.json", "w") as f:
    json.dump(recipes, f, indent=2, ensure_ascii=False)


In [7]:
# remove ingredient group with 0 ingredients
for r in recipes:
    before = len(r["ingredient_groups"])
    r["ingredient_groups"] = [g for g in r["ingredient_groups"] if len(g["ingredients"]) > 0]
    after = len(r["ingredient_groups"])

# Save
with open("recipes_cleaned.json", "w") as f:
    json.dump(recipes, f, indent=2, ensure_ascii=False)

In [8]:
def parse_amount(amount_str, unit_str):
    match = re.match(r'^([\d\s./]+)\s*([a-zA-Z]+)\s*/\s*([\d\s./]+)\s*([a-zA-Z]+)$', amount_str)

    # only keep the first match to keep it metric (pattern is that the first match is the one with metric units)
    if match:
        amount_str = match.group(1).strip()
        unit_str = match.group(2).strip()
        alt_amount = match.group(3).strip()
        alt_unit = match.group(4).strip()

        return amount_str, unit_str, alt_amount, alt_unit
    # if dual amount not found
    return amount_str, unit_str, None, None

def standardise_ingredient_group(ingredient_groups_list):
    for ingredient_group in ingredient_groups_list:

        # strip excess characters from ing group name
        if ingredient_group["group_name"] is not None:
            ingredient_group["group_name"] = ingredient_group["group_name"].rstrip(":").strip().title()

        # clean each ingredient
        for ing in ingredient_group.get("ingredients", []):
            # parse dual units
            amuont, unit, alt_amount, alt_unit = parse_amount(ing["amount"], ing["unit"])
            ing["amount"] = amuont
            ing["unit"] = unit

            if alt_amount:
                ing["alt_amount"] = alt_amount
                ing["alt_unit"] = alt_unit
            
            # clean notes
            if ing.get("notes"):
                note = ing["notes"].lstrip(", ").strip()
                note = note.lstrip(" ").strip()

                # If the entire note is wrapped in parentheses, remove them
                if note.startswith("(") and note.endswith(")") and note.count("(") == 1:
                    note = note[1:-1].strip()
                
                ing["notes"] = note
        
    # handle null groups
    # find all groups in ingredient group list where group name is None
    null_groups = [g for g in ingredient_groups_list if g["group_name"] is None]
    
    if len(null_groups) == 0:
        # nothing to fix
        pass
    
    elif len(ingredient_groups_list) == 1 and len(null_groups) == 1:
        # only one group and it's null assume it's the full recipe
        null_groups[0]["group_name"] = "Full Recipe"
    
    else:
        for g in null_groups:
            g["group_name"] = "Unknown"
    
    return ingredient_groups_list

In [9]:
# normalse group names and remove leading commaas and spaces from notes
with open("recipes_cleaned.json", 'r') as f:
    recipes = json.load(f)

for recipe in recipes:
    standardise_ingredient_group(recipe["ingredient_groups"])

# Save
with open("recipes_cleaned.json", "w") as f:
    json.dump(recipes, f, indent=2, ensure_ascii=False)